In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CWD = Path.cwd()
BASE_DIR = CWD.parent if CWD.name == "analysis" else CWD
RESULT_DIR = BASE_DIR / "results"
DASH_DIR = BASE_DIR / "dashboard"
DASH_DIR.mkdir(parents=True, exist_ok=True)

train_cat = pd.read_csv(RESULT_DIR / "olap_step03_03_train_category.csv").sort_values("mean_delay_min", ascending=False)
zone_df = pd.read_csv(RESULT_DIR / "olap_step03_03_railway_zone.csv")
zone_df = zone_df[zone_df["observed_stops"] >= 10000].sort_values("bottleneck_gt_2sd_pct", ascending=True)
monthly_df = pd.read_csv(RESULT_DIR / "olap_step03_03_temporal_monthly.csv")
monthly_df["label"] = monthly_df["month_name"].str.slice(0, 3) + " " + monthly_df["year"].astype(str)
route_df = pd.read_csv(RESULT_DIR / "olap_step03_04_route_stage_progression.csv")
hubs_df = pd.read_csv(RESULT_DIR / "step04_01_top25_composite_bottleneck_hubs.csv").head(15).iloc[::-1]
model_df = pd.read_csv(RESULT_DIR / "step04_03_model_comparison_overall.csv")

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        "1. Train Category: Mean Delay (min) & Bottleneck (>2 SD %)",
        "2. Railway Zones: Bottleneck Rate (>2 SD %)",
        "3. Top 15 Composite Structural-Operational Bottleneck Hubs",
        "4. Monthly Delay & Bottleneck Trajectory (Feb 2025 – Feb 2026)",
        "5. Route Stage Progression & Terminal Schedule Padding",
        "6. Independent Test Set Model Performance (MAE & RMSE in min)",
    ),
    vertical_spacing=0.11,
    horizontal_spacing=0.10,
)

# 1. Train Category
fig.add_trace(
    go.Bar(x=train_cat["primary_type_code"], y=train_cat["mean_delay_min"], name="Category Mean Delay (min)", marker_color="#2980b9"),
    row=1, col=1
)

# 2. Railway Zones
fig.add_trace(
    go.Bar(x=zone_df["bottleneck_gt_2sd_pct"], y=zone_df["station_zone"], orientation="h", name="Zone Bottleneck >2 SD (%)", marker_color="#c0392b"),
    row=1, col=2
)

# 3. Top 15 Composite Bottleneck Hubs
fig.add_trace(
    go.Bar(
        x=hubs_df["composite_hub_stress_score"],
        y=hubs_df["station_name"] + " (" + hubs_df["station_zone"] + ")",
        orientation="h",
        name="Composite Hub Stress Score",
        marker_color="#8e44ad",
    ),
    row=2, col=1
)

# 4. Monthly Trajectory
fig.add_trace(
    go.Scatter(x=monthly_df["label"], y=monthly_df["mean_delay_min"], mode="lines+markers", name="Monthly Mean Delay (min)", line=dict(color="#d35400", width=3)),
    row=2, col=2
)
fig.add_trace(
    go.Scatter(x=monthly_df["label"], y=monthly_df["median_delay_min"], mode="lines+markers", name="Monthly Median Delay (min)", line=dict(color="#27ae60", width=2, dash="dash")),
    row=2, col=2
)

# 5. Route Stage Progression
fig.add_trace(
    go.Bar(x=route_df["route_stage"], y=route_df["mean_delay_min"], name="Stage Mean Delay (min)", marker_color="#16a085"),
    row=3, col=1
)
# 6. Model Evaluation (exact columns from step04_03_model_comparison_overall.csv: model, test_RMSE, test_MAE)
fig.add_trace(
    go.Bar(x=model_df["model"], y=model_df["test_RMSE"], name="Test RMSE (min)", marker_color="#2c3e50"),
    row=3, col=2
)
fig.add_trace(
    go.Bar(x=model_df["model"], y=model_df["test_MAE"], name="Test MAE (min)", marker_color="#f39c12"),
    row=3, col=2
)

fig.update_layout(
    height=1350,
    width=1450,
    title_text="<b>Indian Railways Data Warehousing, OLAP, Network Bottleneck & Delay Prediction Executive Dashboard</b><br>"
               "<sup>36,614,824 Cleaned Warehouse Rows | 35,582,318 Observed Rows | 33,158,834 Scheduled Transitions | Independent Test Evaluation</sup>",
    template="plotly_white",
    showlegend=True,
)

html_out = DASH_DIR / "indian_railways_dashboard.html"
fig.write_html(str(html_out), include_plotlyjs="cdn")
print(f"Saved interactive HTML dashboard to: {html_out}")

Saved interactive HTML dashboard to: D:\M_Tech_CSE_AI\SEM01\DWDM\project\indian_railway_delay\dashboard\indian_railways_dashboard.html
